# Risk model: does a faster volatility estimate help?

**Question.** The qps risk model forecasts volatility and correlation from the last 36 months of returns, giving every month the same weight. The research backlog flagged that these forecasts run low: realised portfolio volatility is about 1.3x (trend) and 1.4x (STATIC) what the model predicted. Would an exponentially weighted estimate (recent months count more), or a mix of a fast and a slow estimate, forecast better and trade better?

**Answer in one line.** No estimator fixes the under-prediction, and the Sharpe gains from faster estimators are small, in-sample and chosen from six variants, so the current 36-month equal-weight window stays (results in section 4).

Everything here is exploratory, in-sample on 1976-2014, and uses the endpoint-sensitivity data (the only full-period run). Code: `src/risk_model/rolling.py`, `scripts/compare_risk_models.py`, settings in `config/risk_model_comparison.json`.

**Cost revision status (26 October 2026):** configuration now uses 2 bps as primary and 0/1/2/5 bps scenarios. Saved numerical conclusions describe the earlier 5 bps experiment; the [controlled rerun is now complete](../reports/cost_revision_20261026/SUMMARY.md), with dated results below or in that report. Earlier sections retain their historical numerical results.

## 1. The baseline: what the risk model does today

Each month, for the markets that are eligible (complete 36-month history, known trend signal), the model builds an annualised **covariance matrix** `Q` from the previous 36 monthly returns (information through month t-1):

1. **Volatility** `sigma_i`: sample standard deviation of market i's 36 returns times sqrt(12), floored at 2%. Every month in the window has the same weight (1/36).
2. **Correlation** `R`: sample correlation matrix of the same 36 months, then **shrunk toward the identity**: `R_shrunk = 0.5 R + 0.5 I`. This keeps `Q` positive definite and damps noisy correlations.
3. `Q = D R_shrunk D`, with `D = diag(sigma)`.

`Q` is used twice downstream:
- **Sizing**: each market starts at `0.40 / (number of eligible markets x sigma_i)`, so volatile markets get smaller positions. The trend signal sets the sign.
- **Limits**: the portfolio is scaled down if forecast volatility `sqrt(w'Qw)` exceeds 10% or gross exposure exceeds 3x.

**Calibration** is how we judge the forecast: `std( gross return / (forecast volatility / sqrt(12)) )`. If the forecast is right, that standardised return has standard deviation 1. Today it is **1.27 for trend and 1.41 for STATIC**, so realised risk is 27-41% above forecast and the 10% risk ceiling is not really a 10% ceiling.

## 2. What we are trying

Only the **weights on the 36 months** change. Everything else (window length, eligibility, shrinkage 0.5, 2% floor, limits, 10% band, 5 bps cost, the v2 trend signal) is held fixed, so any difference comes from the estimator.

- **EWMA**: month `a` months ago gets weight proportional to `0.5^(a / h)` where `h` is the half-life in months. A short half-life reacts quickly to a volatility spike; a long one approaches the equal-weight baseline.
- **Fast/slow**: average the covariance from a fast and a slow estimator. This reacts to spikes without throwing away the long memory.

Weighted covariance uses the reliability correction `sum w (x - m)(x - m)' / (1 - sum w^2)`, which equals the usual `ddof=1` sample covariance when weights are equal (checked in `tests/risk_model/test_risk_model.py`).

| Name | Half-life(s), months | Reading |
|---|---|---|
| `equal_36m` | equal weights | **baseline** |
| `ewma_hl6` / `hl12` / `hl24` | 6 / 12 / 24 | single decaying estimator, fast to slow |
| `fast_slow_3_equal` | 3 and equal | very fast mixed with the baseline |
| `fast_slow_6_24` | 6 and 24 | two decaying estimators |

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]

from compare_risk_models import compare, load_inputs
from src.risk_model.rolling import decay_weights

import json
VARIANTS = json.loads((ROOT / "config" / "risk_model_comparison.json").read_text())["variants"]
WINDOW = 36

### How the weights differ
Weight on each of the 36 months (right edge = most recent month). The fast/slow variants are the average of their component curves.

In [2]:
fig, ax = plt.subplots(figsize=(8, 3.5))
months_ago = np.arange(WINDOW)[::-1]
for name, half_lives in VARIANTS.items():
    weights = np.mean([decay_weights(WINDOW, h) for h in half_lives], axis=0)
    ax.plot(-months_ago, weights, label=name, lw=2.5 if name == "equal_36m" else 1.5,
            color="black" if name == "equal_36m" else None)
ax.set_xlabel("months before the forecast")
ax.set_ylabel("weight")
ax.set_title("Weight given to each month of the 36-month window")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

/var/folders/2s/4bwc4_c534sg_8zdq7wkq00c0000gn/T/ipykernel_70414/825526853.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Run the comparison
Each estimator is run through the full pipeline (targets, band, limits, costs, simulation) for the trend strategy and its matched STATIC always-long benchmark. About 15 seconds.

In [3]:
INPUTS = load_inputs()
table = compare(INPUTS, VARIANTS)
view = table.rename(columns={"realized_predicted_ratio": "calibration"})
view.pivot(index="estimator", columns="strategy", values=["sharpe", "calibration"]).loc[list(VARIANTS)].round(3)

sharpe        calibration       
strategy          static  trend      static  trend
estimator                                         
equal_36m          0.484  1.071       1.406  1.272
ewma_hl6           0.461  1.118       1.447  1.312
ewma_hl12          0.476  1.096       1.402  1.260
ewma_hl24          0.481  1.083       1.396  1.255
fast_slow_3_equal  0.466  1.114       1.411  1.288
fast_slow_6_24     0.473  1.103       1.404  1.269

## 4. Results
Trend strategy, 10% band, 5 bps. **Calibration** = std of return over forecast volatility (1.0 is perfect; above 1 means risk was under-forecast). Differences from the baseline in the last two columns.

In [4]:
trend = table[table.strategy == "trend"].set_index("estimator").loc[list(VARIANTS)]
base_row = trend.loc["equal_36m"]
result = pd.DataFrame({
    "Sharpe": trend.sharpe,
    "CAGR": trend.cagr,
    "Worst drawdown": trend.max_drawdown,
    "Turnover (x/yr)": trend.turnover_ann,
    "Cost (%/yr)": trend.mean_cost_ann * 100,
    "Calibration": trend.realized_predicted_ratio,
    "Sharpe vs baseline": trend.sharpe - base_row.sharpe,
    "Calibration vs baseline": trend.realized_predicted_ratio - base_row.realized_predicted_ratio,
})
result.round(3)

,Sharpe,CAGR,Worst drawdown,Turnover (x/yr),Cost (%/yr),Calibration,Sharpe vs baseline,Calibration vs baseline
estimator,,,,,,,,
equal_36m,1.071,0.111,-0.145,7.409,0.370,1.272,0.000,0.000
ewma_hl6,1.118,0.118,-0.164,7.808,0.390,1.312,0.047,0.040
ewma_hl12,1.096,0.113,-0.154,7.516,0.376,1.260,0.025,-0.012
ewma_hl24,1.083,0.112,-0.146,7.435,0.372,1.255,0.012,-0.017
fast_slow_3_equal,1.114,0.116,-0.144,7.713,0.386,1.288,0.043,0.016
fast_slow_6_24,1.103,0.114,-0.152,7.564,0.378,1.269,0.032,-0.004


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, column, title, line in [
    (axes[0], "sharpe", "Trend Sharpe (higher is better)", None),
    (axes[1], "realized_predicted_ratio", "Calibration (1.0 = forecast risk is right)", 1.0),
]:
    for strategy, colour in [("trend", "tab:blue"), ("static", "tab:orange")]:
        frame = table[table.strategy == strategy].set_index("estimator").loc[list(VARIANTS)]
        ax.barh(np.arange(len(VARIANTS)) + (0.2 if strategy == "static" else -0.2), frame[column],
                height=0.4, color=colour, label=strategy)
    ax.set_yticks(range(len(VARIANTS)), list(VARIANTS))
    ax.invert_yaxis()
    ax.set_title(title)
    if line:
        ax.axvline(line, color="black", ls="--")
axes[0].legend()
plt.tight_layout()
plt.show()

/var/folders/2s/4bwc4_c534sg_8zdq7wkq00c0000gn/T/ipykernel_70414/3959762768.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Why is calibration off? A diagnostic on the shrinkage

No estimator moved calibration, so the estimation window is not the cause. One candidate is the **correlation shrinkage**: pulling correlations toward zero makes the portfolio look more diversified than it is, which understates forecast volatility, most of all in crises when correlations jump. The task was to keep shrinkage, so this is a diagnostic only, with the baseline estimator (equal weights) and less shrinkage (smaller number = less shrinkage; 0.5 is the baseline).

In [6]:
rows = []
for shrinkage in [0.5, 0.25, 0.1]:
    settings = {**INPUTS[0], "correlation_shrinkage": shrinkage}
    out = compare(INPUTS, {"equal_36m": [None]}, settings)
    rows.append(out.assign(shrinkage=shrinkage))
diagnostic = pd.concat(rows)
diagnostic.pivot(index="shrinkage", columns="strategy", values=["realized_predicted_ratio", "sharpe"]).round(3)

realized_predicted_ratio        sharpe       
strategy                    static  trend static  trend
shrinkage                                              
0.10                         1.148  1.095  0.487  1.085
0.25                         1.227  1.150  0.485  1.078
0.50                         1.406  1.272  0.484  1.071

## 6. Reading the results

- **The estimator is not the calibration problem.** Calibration stays between 1.26 and 1.31 for trend and 1.40 to 1.45 for STATIC across all six estimators. Changing how fast the window forgets does not move it.
- **Faster estimators raise the trend Sharpe by 0.01 to 0.05** (1.071 to as much as 1.118), with a little more turnover and, for the fastest, a deeper worst drawdown (-16.4% at a 6-month half-life vs -14.5%). STATIC gets slightly worse. The gain is small next to the noise in a 39-year Sharpe, and it is the best of six variants tried on the same data, so it is not evidence of a better model.
- **Shrinkage is the lever on calibration.** With less shrinkage (0.1 instead of 0.5) trend calibration falls from 1.27 to 1.10 and STATIC from 1.41 to 1.15, with trend Sharpe up slightly (1.071 to 1.085) and STATIC unchanged. This fits the story that shrinking correlations toward zero understates portfolio risk. It is a diagnostic: the shrinkage was meant to stay fixed here, and a lower value would need its own robustness check (positive definiteness is guaranteed by the 2% floor and the shrunk matrix, but estimation noise rises with fewer shrinkage).

**Decision.** Keep the 36-month equal-weight window. If anything is changed next, test the shrinkage level (and a calibration-based rescale of the risk ceiling) rather than the estimator, and confirm on data outside 1976-2014.

## Cost revision update — 26 October 2026

The earlier sections retain the 5 bps experiment. The following table comes from the separately identified `cost_revision_20261026` run at the same 10% band. No estimator has been promoted.

| Estimator | TREND Sharpe at 1 bps | TREND Sharpe at 2 bps | Calibration at 2 bps |
|---|---:|---:|---:|
| equal_36m | 1.099 | 1.092 | 1.272 |
| ewma_hl6 | 1.147 | 1.140 | 1.312 |
| ewma_hl12 | 1.125 | 1.118 | 1.260 |
| ewma_hl24 | 1.112 | 1.105 | 1.255 |
| fast_slow_3_equal | 1.144 | 1.136 | 1.288 |
| fast_slow_6_24 | 1.132 | 1.124 | 1.269 |

The calibration shortfall remains. These are exploratory, inspected historical comparisons, not independent validation. [Full table with STATIC](../reports/cost_revision_20261026/output/risk_model_comparison/cost_comparison.csv) · [Provenance and checks](../reports/cost_revision_20261026/SUMMARY.md).